In [2]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
from verimon.logger import setup_logging
setup_logging()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
from verimon import loaders
from verimon.transformations import simulator_unroll
from stormvogel.mapping import stormvogel_to_stormpy

mc_sl_2x2 = "../tests/snake_ladder/mc_2x2.pm"
mon_sl_2x2 = "../tests/snake_ladder/mon_2x2.pm"

mc_sl_2x2_deadlock = "../tests/snake_ladder/mc_2x2_deadlock.pm"
mon_sl_2x2_deadlock = "../tests/snake_ladder/mon_2x2_deadlock.pm"

mc_sl_nxn = "../tests/snake_ladder/mc_nxn.pm"
mon_sl_nxn = "../tests/snake_ladder/mon_nxn.pm"

mc_sl_u_nxn = "../tests/snake_ladder/mc_u_nxn.pm"

mc_simple = "../tests/simple/mc1.pm"
mon_simple = "../tests/simple/mon1.pm"

# mc, n, ladders, snakes = loaders.load_defined_snl(mc_sl_nxn)
mon_sv = loaders.load_dfa(mon_sl_nxn)
mon_sv = simulator_unroll(mon_sv, 5)
mon = stormvogel_to_stormpy(mon_sv)

In [7]:
n, ladders, snakes = loaders.random_snl_board(2**2)

# Random snakes and ladders
mc = loaders.load_snl_stormpy(mc_sl_u_nxn, n, ladders, snakes)

milton_snakes = {98: 76, 95: 75, 93: 73, 87: 24, 64: 60, 62: 19, 55: 53, 49: 11, 47: 26, 16: 6}
milton_ladders = {1: 38, 4: 14, 9: 31, 28: 64, 40: 42, 36: 44, 51: 67, 71: 91, 80: 100}
# mc = loaders.load_snl_stormpy(mc_sl_u_nxn, n := 10**2, ladders:=milton_ladders, snakes:=milton_snakes)

In [10]:
from stormvogel.mapping import stormpy_to_stormvogel
from stormvogel.show import show

mc_sv = stormpy_to_stormvogel(mc)
loaders._add_valuation_to_sv_labels(mc, mc_sv)
show(mc_sv)
show(mon_sv)

Output()

Output()

Output()

Output()

In [11]:
from verimon.generator import Verifier
from time import time

t = time()
pomdp = Verifier(mc, mon, "good")
# pomdp.apply_spec('P>=0.5 [ F<=3 "good"]')
pomdp.create_product()
print(f"--------------------\nStarting Paynt after {time() - t}s")
assignment = pomdp.check_paynt_prop('Pmax=? [ (F "goal") ]')

Observation for mon: [l=0], init, step=0,  mc: init, 
added observation
Observation for mon: [l=1], accepting, step=1,  mc: normal, 
added observation
Observation for mon: [l=1], accepting, step=1,  mc: good, normal, 
Observation for mon: [l=1], accepting, step=1,  mc: ladder, 
Observation for mon: [l=1], accepting, step=1,  mc: snake, 
Observation for mon: [l=1], accepting, step=2,  mc: ladder, 
added observation
Observation for mon: [l=1], accepting, step=2,  mc: snake, 
Observation for mon: [l=1], accepting, step=2,  mc: normal, 
Observation for mon: [l=1], accepting, step=2,  mc: good, normal, 
Observation for mon: [l=1], accepting, step=3,  mc: ladder, 
added observation
Observation for mon: [l=1], accepting, step=3,  mc: snake, 
Observation for mon: [l=1], accepting, step=3,  mc: normal, 
Observation for mon: [l=1], accepting, step=3,  mc: good, normal, 
Observation for mon: [l=1], accepting, horizon, step=4,  mc: ladder, 
added observation
Observation for mon: [l=1], accepting, 

In [18]:
psv = stormpy_to_stormvogel(pomdp.pomdp)
loaders._add_valuation_to_sv_labels(pomdp.pomdp, psv)
show(psv)

Output()

Output()

In [12]:
%matplotlib notebook
from verimon.draw import animate_player_movement
import math
from IPython.display import HTML

poss = pomdp.simulate_paynt_assignment(assignment, 10000)

goal_squares = [n]
player_path = poss

animation = animate_player_movement(int(math.sqrt(n)), snakes, ladders, goal_squares, player_path)
HTML(animation.to_jshtml())

INFO:2024-10-30 13:10:13,162 - (11.90s) - generator.py - s2, obs=2, labels=init 
INFO:2024-10-30 13:10:13,165 - (0.00s) - generator.py - --[2, normal]-->	s3, val=[pos=2], labels=
--[3, normal]-->	s9, val=[pos=2], labels=
--[4, normal]-->	s14, val=[pos=4], labels=
--[5, normal]-->	s17, val=[pos=4], labels=
--[6, end]-->	s0, val=[pos=-1], labels=goal 
INFO:2024-10-30 13:10:13,166 - (0.00s) - generator.py - it took 1 tries until the goal was reached 


<IPython.core.display.Javascript object>

In [ ]:
from stormvogel.show import show
from stormvogel.mapping import stormpy_to_stormvogel
from stormpy import model_checking, parse_properties

induced_mc = pomdp.created_induced_mc(assignment)
imc =  stormpy_to_stormvogel(induced_mc)
result_goal = model_checking(induced_mc, parse_properties('Pmax=? [F "goal"]')[0])
result_stop = model_checking(induced_mc, parse_properties('Pmax=? [F "stop"]')[0])
prop_goal = result_goal.at(induced_mc.initial_states[0])
prop_stop = result_stop.at(induced_mc.initial_states[0])
print(f"probability to reach goal={prop_goal} and stop={prop_stop}. Total={prop_goal+prop_stop}")
show(imc)